In [1]:
import pymysql
import pandas as pd
import numpy as np
import re
from DATA.stock_invest_function import get_db_host

In [2]:
def fetch_fs_data_by_ticker(db_info: dict,
                            ticker: str,
                            table_name: str = "korea_fs_data_from_DART") -> pd.DataFrame:
    """
    특정 ticker의 재무제표 데이터를 DB에서 조회.
    ticker가 존재하지 않을 경우 메시지 출력 후 빈 DataFrame 반환.
    """

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4"
    )

    try:
        # 먼저 ticker 존재 여부 확인
        check_sql = f"SELECT COUNT(*) AS cnt FROM {table_name} WHERE ticker = %s"
        with conn.cursor() as cur:
            cur.execute(check_sql, (ticker,))
            result = cur.fetchone()
            cnt = result[0]

        if cnt == 0:
            print(f"[INFO] ticker '{ticker}' 는(은) 데이터베이스에 존재하지 않습니다.")
            return pd.DataFrame()   # 빈 DF 반환

        # ticker 존재 → 실제 데이터 조회
        query = f"""
            SELECT
                corp_code,
                bsns_year,
                reprt_code,
                quarter,
                account_id,
                sj_div,
                sj_nm,
                account_nm,
                thstrm_nm,
                thstrm_amount,
                report_date,
                ticker
            FROM {table_name}
            WHERE ticker = %s
            ORDER BY
                bsns_year,
                reprt_code,
                sj_div,
                account_nm
        """

        df = pd.read_sql(query, conn, params=[ticker])
        return df

    finally:
        conn.close()

def adjust_quarterly_from_index(df: pd.DataFrame, value_cols: list) -> pd.DataFrame:
    df = df.copy()
    df.index = pd.to_datetime(df.index)
    original_index_name = df.index.name

    df["year"] = df.index.year
    df["quarter"] = df.index.month.map({3: "Q1", 6: "Q2", 9: "Q3", 12: "Q4"})

    adjusted_chunks = []

    for year, grp in df.groupby("year"):
        grp = grp.sort_index()

        q1 = grp[grp["quarter"] == "Q1"]
        q2 = grp[grp["quarter"] == "Q2"]
        q3 = grp[grp["quarter"] == "Q3"]
        q4 = grp[grp["quarter"] == "Q4"]

        if len(q4) > 0:
            q4 = q4.copy()

            for col in value_cols:
                if col not in grp.columns:
                    continue

                fy = q4[col].iloc[0]  # 12월 값(FY라고 가정)
                if pd.isna(fy):
                    continue

                prev_sum = (
                    q1[col].fillna(0).sum()
                    + q2[col].fillna(0).sum()
                    + q3[col].fillna(0).sum()
                )

                q4[col] = fy - prev_sum   # ★ 여기서 딱 Q4만 수정

            adjusted_chunks.extend([q1, q2, q3, q4])
        else:
            # 4Q(12월)가 없으면 그 연도는 그대로
            adjusted_chunks.append(grp)

    result = pd.concat(adjusted_chunks).sort_index()
    result = result.drop(columns=["year", "quarter"])
    result.index.name = original_index_name
    return result


from typing import Optional, List

def cumulative_to_quarterly(df: pd.DataFrame,
                            value_cols: List[str],
                            exclude_date: Optional[str] = "2025-12-31") -> pd.DataFrame:
    """
    연도별 누적값(1Q,2Q,3Q,4Q)을 순수 분기값으로 변환.
    df: index가 날짜(분기말)인 DataFrame
    value_cols: 변환할 숫자 컬럼 리스트
    exclude_date: 제외할 날짜 (예: '2025-12-31')
    """
    out = df.copy()
    out.index = pd.to_datetime(out.index)
    out = out.sort_index()

    # 특정 날짜 제거
    if exclude_date is not None:
        out = out.loc[out.index != pd.to_datetime(exclude_date)].copy()

    years = out.index.year

    for col in value_cols:
        def _to_quarterly(s: pd.Series) -> pd.Series:
            s = s.sort_index()
            q = s.diff()
            if len(s) > 0:
                q.iloc[0] = s.iloc[0]
            return q

        out[col] = (
            out[col]
            .groupby(years)
            .apply(_to_quarterly)
            .reset_index(level=0, drop=True)
        )

    return out


# 0으로 나누는 경우 inf가 생기지 않도록 float 변환 + 나누기 후 정리
def safe_divide(num, den):
    result = num / den
    # 0으로 나눠서 생긴 inf/-inf 를 NaN으로 처리
    result = result.replace([np.inf, -np.inf], np.nan)
    return result



account_groups = {

    # -------------------------------------------------
    # 1) 매출액 (Revenue)
    # -------------------------------------------------
    "revenue": [
        "ifrs_Revenue",
        "ifrs-full_Revenue",
    ],

    # -------------------------------------------------
    # 2) 매출총이익 (Gross Profit)
    # -------------------------------------------------
    "gross_profit": [
        "ifrs_GrossProfit",
        "ifrs-full_GrossProfit",
    ],

    # -------------------------------------------------
    # 3) 영업이익 (Operating Income)
    # -------------------------------------------------
    "operating_income": [
        "dart_OperatingIncomeLoss",
    ],

    # -------------------------------------------------
    # 4) 당기순이익 (Net Income) - 총당기순이익
    #     손익계산서 기준 전체 당기순이익 + CF용 당기순이익 포함
    # -------------------------------------------------
    "net_income_total": [
        "ifrs_ProfitLoss",
        "ifrs-full_ProfitLoss",
        "dart_ProfitLossForStatementOfCashFlows",
    ],

    # -------------------------------------------------
    # 4-1) 계속사업 관련 손익 (계속영업이익, 계속사업 법인세 등)
    # -------------------------------------------------
    "continuing_operations": [
        "ifrs_ProfitLossBeforeTax",
        "ifrs-full_ProfitLossBeforeTax",
    ],

    # -------------------------------------------------
    # 4-2) 계속사업 법인세 등
    # -------------------------------------------------
    "income_tax": [
        "ifrs_IncomeTaxExpenseContinuingOperations",             # 계속사업 법인세비용
        "ifrs-full_IncomeTaxExpenseContinuingOperations",        # (full IFRS) 계속사업 법인세비용
    ],


    # -------------------------------------------------
    # 5) 당기순이익 - 지배기업 소유주 귀속
    # -------------------------------------------------
    "net_income_parent": [
        "ifrs_ProfitLossAttributableToOwnersOfParent",
        "ifrs-full_ProfitLossAttributableToOwnersOfParent",
    ],

    # -------------------------------------------------
    # 6) 당기순이익 - 비지배지분 귀속
    # -------------------------------------------------
    "net_income_nci": [
        "ifrs_ProfitLossAttributableToNoncontrollingInterests",
        "ifrs-full_ProfitLossAttributableToNoncontrollingInterests",
    ],

    "discontinued_pl_accounts" : ["ifrs_ProfitLossFromDiscontinuedOperations"],
}

account_groups_bs = {

    # -------------------------------------------------
    # 1) 자산총계 (Total Assets)
    # -------------------------------------------------
    "assets_total": [
        "ifrs_Assets",              # IFRS 전체 자산
        "ifrs-full_Assets",         # IFRS full
    ],

    "cash": [
        "ifrs_CashAndCashEquivalents",
        "ifrs-full_CashAndCashEquivalents",              # IFRS 현금
    ],

    # -------------------------------------------------
    # 2) 유동자산 (Current Assets)
    # -------------------------------------------------
    "current_assets": [
        "ifrs_CurrentAssets",
        "ifrs-full_CurrentAssets",
    ],

    # -------------------------------------------------
    # 3) 재고자산 (Inventories)
    # -------------------------------------------------
    "inventories": [
        "ifrs_Inventories",
        "ifrs-full_Inventories",
    ],

    # -------------------------------------------------
    # 4) 유형자산 (Property, Plant and Equipment)
    # -------------------------------------------------
    "ppe": [
        "ifrs_PropertyPlantAndEquipment",
        "ifrs-full_PropertyPlantAndEquipment",
    ],

    # -------------------------------------------------
    # 5) 부채총계 (Total Liabilities)
    # -------------------------------------------------
    "liabilities_total": [
        "ifrs_Liabilities",
        "ifrs-full_Liabilities",
    ],

    # -------------------------------------------------
    # 6) 유동부채 (Current Liabilities)
    # -------------------------------------------------
    "current_liabilities": [
        "ifrs_CurrentLiabilities",
        "ifrs-full_CurrentLiabilities",
    ],

    # -------------------------------------------------
    # 7) 비유동부채 (Noncurrent Liabilities)
    # -------------------------------------------------
    "noncurrent_liabilities": [
        "ifrs_NoncurrentLiabilities",
        "ifrs-full_NoncurrentLiabilities",
    ],
}

account_groups_cf = {

    # -------------------------------------------------
    # 1) 영업활동 현금흐름
    # -------------------------------------------------
    "cf_operating": [
        "ifrs_CashFlowsFromUsedInOperatingActivities",
        "ifrs-full_CashFlowsFromUsedInOperatingActivities",
    ],

    # 영업활동 조정항목 (당기순이익 → 영업CF로 reconcile)
    "cf_adjustments": [
        "ifrs_AdjustmentsForReconcileProfitLoss",
        "ifrs-full_CashFlowsFromUsedInOperations",  # Operations 기준도 함께
    ],

    # -------------------------------------------------
    # 2) 투자활동 현금흐름
    # -------------------------------------------------
    "cf_investing": [
        "ifrs_CashFlowsFromUsedInInvestingActivities",
        "ifrs-full_CashFlowsFromUsedInInvestingActivities",
    ],

    # -------------------------------------------------
    # 3) 재무활동 현금흐름
    # -------------------------------------------------
    "cf_financing": [
        "ifrs_CashFlowsFromUsedInFinancingActivities",
        "ifrs-full_CashFlowsFromUsedInFinancingActivities",
    ],

    # -------------------------------------------------
    # 4) 법인세 납부(환급) - 영업활동 분류
    # -------------------------------------------------
    "cf_tax_operating": [
        "ifrs_IncomeTaxesPaidRefundClassifiedAsOperatingActivities",
        "ifrs-full_IncomeTaxesPaidRefundClassifiedAsOperatingActivities",
    ],

    # -------------------------------------------------
    # 5) 이자수익/이자비용 - 영업활동 분류
    # -------------------------------------------------
    "cf_interest_received": [
        "ifrs_InterestReceivedClassifiedAsOperatingActivities",
        "ifrs-full_InterestReceivedClassifiedAsOperatingActivities",
    ],
    "cf_interest_paid": [
        "ifrs_InterestPaidClassifiedAsOperatingActivities",
        "ifrs-full_InterestPaidClassifiedAsOperatingActivities",
    ],

    # -------------------------------------------------
    # 6) 배당수익/배당금 지급
    # -------------------------------------------------
    "cf_dividends_received": [
        "ifrs_DividendsReceivedClassifiedAsOperatingActivities",
        "ifrs-full_DividendsReceivedClassifiedAsOperatingActivities",
    ],
    "cf_dividends_paid": [
        "ifrs_DividendsPaidClassifiedAsFinancingActivities",
        "ifrs-full_DividendsPaidClassifiedAsFinancingActivities",
        "ifrs_DividendsPaid",
        "ifrs-full_DividendsPaid",
    ],

    # -------------------------------------------------
    # 7) 차입/상환 (재무활동 디테일)
    # -------------------------------------------------
    "cf_borrowings": [
        "ifrs_ProceedsFromBorrowingsClassifiedAsFinancingActivities",
        "ifrs-full_ProceedsFromBorrowingsClassifiedAsFinancingActivities",
    ],
    "cf_repayments": [
        "ifrs_RepaymentsOfBorrowingsClassifiedAsFinancingActivities",
        "ifrs-full_RepaymentsOfBorrowingsClassifiedAsFinancingActivities",
    ],

    # -------------------------------------------------
    # 8) 기초/기말 현금 및 현금성자산, 증감, 환율효과
    # -------------------------------------------------
    "cf_beginning_cash": [
        "dart_CashAndCashEquivalentsAtBeginningOfPeriodCf",
    ],
    "cf_ending_cash": [
        "dart_CashAndCashEquivalentsAtEndOfPeriodCf",
    ],
    "cf_increase_decrease_cash": [
        "ifrs_IncreaseDecreaseInCashAndCashEquivalents",
        "ifrs-full_IncreaseDecreaseInCashAndCashEquivalents",
    ],
    "cf_fx_effect": [
        "ifrs_EffectOfExchangeRateChangesOnCashAndCashEquivalents",
        "ifrs-full_EffectOfExchangeRateChangesOnCashAndCashEquivalents",
    ],

    # -------------------------------------------------
    # 9) 기타: 단기예금/투자, 리스상환, 정부보조금 등
    #    (필요시 나중에 세분화해서 쓰실 수 있게 모아둠)
    # -------------------------------------------------
    "cf_other_investing": [
        "ifrs-full_ProceedsFromSalesOfPropertyPlantAndEquipmentClassifiedAsInvestingActivities",
        "ifrs-full_PurchaseOfPropertyPlantAndEquipmentClassifiedAsInvestingActivities",
        "ifrs-full_ProceedsFromSalesOfIntangibleAssetsClassifiedAsInvestingActivities",
        "ifrs-full_PurchaseOfIntangibleAssetsClassifiedAsInvestingActivities",
        "ifrs-full_ProceedsFromGovernmentGrantsClassifiedAsInvestingActivities",
        "ifrs-full_OtherInflowsOutflowsOfCashClassifiedAsInvestingActivities",
    ],
    "cf_other_financing": [
        "ifrs-full_PaymentsOfFinanceLeaseLiabilitiesClassifiedAsFinancingActivities",
        "ifrs-full_PaymentsOfLeaseLiabilitiesClassifiedAsFinancingActivities",
        "ifrs-full_OtherInflowsOutflowsOfCashClassifiedAsFinancingActivities",
        "ifrs-full_SaleOrIssueOfTreasuryShares",
        "ifrs-full_IncreaseDecreaseThroughSharebasedPaymentTransactions",
    ],
}

def extract_is_group(df: pd.DataFrame,
                     group_name: str,
                     account_groups: dict) -> pd.DataFrame:
    """
    손익계산서(IS)에서 account_groups[group_name] 에 속하는
    account_id 행만 추출해서 반환.
    """
    ids = account_groups.get(group_name, [])
    if not ids:
        raise ValueError(f"{group_name} 에 대한 account_id 리스트가 비어 있습니다.")

    mask = ((df["sj_div"] == "IS") | (df["sj_div"] == "CIS")) & df["account_id"].isin(ids)
    out = df.loc[mask].copy()
    return out


def extract_bs_group(df: pd.DataFrame,
                     group_name: str,
                     account_groups_bs: dict) -> pd.DataFrame:

    ids = account_groups_bs.get(group_name, [])
    if not ids:
        raise ValueError(f"{group_name} 에 대한 account_id 리스트가 비어 있습니다.")

    mask = (df["sj_div"] == "BS") & df["account_id"].isin(ids)
    return df.loc[mask].copy()

def extract_cf_group(df: pd.DataFrame,
                     group_name: str,
                     account_groups_cf: dict) -> pd.DataFrame:
    """
    현금흐름표(CF)에서 account_groups_cf[group_name] 에 속하는
    account_id 행만 추출해서 반환.
    """
    ids = account_groups_cf.get(group_name, [])
    if not ids:
        raise ValueError(f"{group_name} 에 대한 account_id 리스트가 비어 있습니다.")

    mask = (df["sj_div"] == "CF") & df["account_id"].isin(ids)
    out = df.loc[mask].copy()
    return out

def normalize_name(name: str) -> str:
    """
    계정명을 정규화하여 유사 항목을 하나의 대표 이름으로 통합하는 함수.
    - 공백 제거
    - 특수문자 제거
    - 숫자 제거
    - 동의어/변형어 통합
    """
    if pd.isna(name):
        return "unknown"

    n = str(name).strip()

    # 1) 특수문자·공백 제거
    n = re.sub(r"[\s\(\)\[\]\/]", "", n)

    # 2) 숫자 제거
    n = re.sub(r"[0-9]+", "", n)

    # 3) 주요 패턴 통합
    replacements = {
        "배당금지급": "배당금지급",
        "배당금의지급": "배당금지급",
        "배당의지급": "배당금지급",
        "배당금": "배당금지급",

        "영업이익손실": "영업이익",
        "영업이익": "영업이익",

        "법인세비용차감전순이익손실": "법인세차감전순이익",
        "법인세비용차감전순이익": "법인세차감전순이익",
        "법인세차감전순이익": "법인세차감전순이익",
    }

    for key, val in replacements.items():
        if key in n:
            return val

    return n  # 기본값


def make_pivot(df: pd.DataFrame, target_name: str) -> pd.DataFrame:
    """
    - 다양한 account_nm이 존재해도 자동으로 하나의 컬럼으로 통합
    - report_date → index
    - thstrm_amount → values
    - target_name: 최종적으로 부여할 표준화된 컬럼 이름
        예: "매출액", "매출총이익", "영업이익", "자산총계", "부채총계", ...
    """

    if df.empty:
        return pd.DataFrame()

    df = df.copy()
    df["report_date"] = pd.to_datetime(df["report_date"])

    # ---------------------------------------------------
    # 1) 계정명 정규화 (공백 제거, 괄호 제거 등)
    # ---------------------------------------------------
    def norm(x):
        x = str(x)
        x = re.sub(r"\s+", "", x)
        x = re.sub(r"\(.*?\)", "", x)
        x = re.sub(r"[^가-힣A-Za-z]", "", x)
        return x

    df["account_nm_rep"] = df["account_nm"].apply(norm)

    # ---------------------------------------------------
    # 2) pivot 생성 — 컬럼이 여러 개 생길 수 있음
    # ---------------------------------------------------
    pivot_df = df.pivot_table(
        index="report_date",
        columns="account_nm_rep",
        values="thstrm_amount",
        aggfunc="sum"
    ).sort_index()

    # 컬럼 리스트
    cols = pivot_df.columns.tolist()

    # ---------------------------------------------------
    # 3) 컬럼이 1개면 바로 rename
    # ---------------------------------------------------
    if len(cols) == 1:
        pivot_df.columns = [target_name]
        return pivot_df

    # ---------------------------------------------------
    # 4) 여러 컬럼이 생긴 경우 자동 병합
    # ---------------------------------------------------
    # ① 새로운 빈 통합 컬럼 생성
    merged = pd.Series(index=pivot_df.index, dtype='float64')

    for c in cols:
        # NaN이 아닌 값 우선 적용
        merged = merged.combine_first(pivot_df[c])

    # ② 모든 컬럼 값 합산 버전도 고려 (의미 있을 경우)
    sum_col = pivot_df.sum(axis=1)

    # NaN이 많지 않은 쪽을 선택
    if sum_col.notna().sum() >= merged.notna().sum():
        merged_final = sum_col
    else:
        merged_final = merged

    # ---------------------------------------------------
    # 5) 최종 1개 컬럼 DataFrame 생성
    # ---------------------------------------------------
    result = pd.DataFrame({target_name: merged_final})
    return result

def adjust_quarterly_q4_only(df: pd.DataFrame, value_cols: list) -> pd.DataFrame:
    """
    1Q, 2Q, 3Q는 원래 값 그대로 두고,
    4Q(12월)만 `FY - (Q1+Q2+Q3)`로 조정하는 함수.

    전제:
      - df.index: DatetimeIndex (분기말 날짜)
      - value_cols: 조정할 수치 칼럼 리스트
    """
    out = df.copy()
    out.index = pd.to_datetime(out.index)

    # 연도별로 처리
    for year in sorted(out.index.year.unique()):
        mask_year = out.index.year == year
        sub = out.loc[mask_year].sort_index()

        q1_idx = sub.index[sub.index.month == 3]
        q2_idx = sub.index[sub.index.month == 6]
        q3_idx = sub.index[sub.index.month == 9]
        q4_idx = sub.index[sub.index.month == 12]

        # 4분기가 없으면 그 해는 스킵
        if len(q4_idx) == 0:
            continue

        # 4분기 행(여러 개라면 첫 번째만 사용한다고 가정)
        q4_i = q4_idx[0]

        for col in value_cols:
            if col not in out.columns:
                continue

            fy = out.loc[q4_i, col]
            if pd.isna(fy):
                continue

            prev_sum = (
                out.loc[q1_idx, col].fillna(0).sum()
                + out.loc[q2_idx, col].fillna(0).sum()
                + out.loc[q3_idx, col].fillna(0).sum()
            )

            # ✅ 오직 4분기 값만 조정
            out.loc[q4_i, col] = fy - prev_sum

    return out

from difflib import SequenceMatcher

def merge_similar_columns_smart(df: pd.DataFrame, threshold: float = 0.75) -> pd.DataFrame:
    """
    pivot table에서 비슷한 계정명(예: 'A', 'A(손실)') 또는
    한국어 계정명이 유사한 경우 자동 병합하여 하나의 컬럼으로 통일.

    threshold : 0~1, 문자열 유사도 임계값
    """
    out = df.copy()

    # 1) 전처리 후 key 생성
    def normalize(col):
        col = str(col)
        col = re.sub(r"\(.*?\)", "", col)   # 괄호 제거
        col = col.replace(" ", "")          # 공백 제거
        col = re.sub(r"[^가-힣A-Za-z]", "", col)   # 특수문자 제거
        return col

    columns = list(out.columns)
    norm_cols = [normalize(c) for c in columns]

    # 2) 그룹핑 수행
    groups = {}
    used = set()

    for i, base in enumerate(norm_cols):
        if i in used:
            continue

        groups[columns[i]] = [columns[i]]
        used.add(i)

        # 다른 컬럼들과 유사도 검사
        for j in range(i + 1, len(columns)):
            if j in used:
                continue

            ratio = SequenceMatcher(None, base, norm_cols[j]).ratio()
            if ratio >= threshold:
                groups[columns[i]].append(columns[j])
                used.add(j)

    # 3) 그룹 내부 병합
    for base_col, cols in groups.items():
        for col in cols:
            if col == base_col:
                continue
            out[base_col] = out[base_col].combine_first(out[col])
            out = out.drop(columns=[col])

    return out


def fetch_fs_from_db(db_info: dict,
                     symbol: str,
                     table_name: str = "korea_fs_data") -> pd.DataFrame:
    """
    MariaDB의 korea_fs_data 테이블에서
    특정 종목(symbol)의 재무 데이터를 읽어오는 함수.

    Parameters
    ----------
    db_info : dict
        {
            "host": ...,
            "port": ...,
            "user": ...,
            "password": ...,
            "database": ...
        }
    symbol : str
        예) 'A000030'
    table_name : str
        기본값 'korea_fs_data'

    Returns
    -------
    pd.DataFrame
        symbol, company_name, date, indicator, value 컬럼 포함.
        데이터가 없으면 빈 DataFrame 반환.
    """

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4",
        autocommit=False,
    )

    try:
        sql = f"""
            SELECT
                symbol,
                company_name,
                date,
                indicator,
                value
            FROM {table_name}
            WHERE symbol = %s
            ORDER BY date, indicator
        """
        df = pd.read_sql(sql, conn, params=(symbol,))
        return df

    finally:
        conn.close()

def df_to_long_format(df: pd.DataFrame, ticker: str) -> pd.DataFrame:
    """
    wide-format 재무 데이터를 long-format(date, ticker, indicator, value) 형태로 변환
    """
    if df.empty:
        raise ValueError("입력된 DataFrame(df)이 비어 있습니다.")

    # 1) index를 date로 사용하기 위해 reset_index
    df2 = df.copy().reset_index()

    # 2) long-format 변환 (melt)
    long_df = pd.melt(
        df2,
        id_vars=["date"],                    # 날짜는 고정
        var_name="indicator",                # 기존 컬럼명이 indicator
        value_name="value"                   # 값은 value
    )

    # 3) ticker 입력값 추가
    long_df["ticker"] = ticker

    # 4) 컬럼 정렬
    long_df = long_df[["date", "ticker", "indicator", "value"]]

    return long_df



def get_unique_tickers(db_info: dict,
                       table_name: str = "korea_fs_data_from_DART") -> list:
    """
    DART 재무데이터 테이블에서 distinct ticker 리스트를 가져오는 함수.
    """
    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4"
    )

    try:
        sql = f"SELECT DISTINCT ticker FROM {table_name} WHERE ticker IS NOT NULL ORDER BY ticker"
        tickers_df = pd.read_sql(sql, conn)
        return tickers_df["ticker"].astype(str).tolist()
    finally:
        conn.close()

def make_fs_pivot(df: pd.DataFrame, item_list: list) -> pd.DataFrame:
    """
    korea_fs_data 형태(df)에서 item_list에 포함된 indicator만 추출하여
    date(index) × indicator(columns) pivot_table 생성.

    Parameters
    ----------
    df : DataFrame (columns: symbol, company_name, date, indicator, value)
    item_list : list of indicators to include

    Returns
    -------
    pivot_df : DataFrame (index=date, columns=item_list)
    """

    if df.empty:
        raise ValueError("입력 df가 비어 있습니다.")

    # 날짜 변환
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])

    # item_list 항목만 필터링
    df_filtered = df[df["indicator"].isin(item_list)].copy()

    if df_filtered.empty:
        print("⚠️ item_list에 해당하는 데이터가 없습니다.")
        return pd.DataFrame()

    # pivot 생성
    pivot_df = df_filtered.pivot_table(
        index="date",
        columns="indicator",
        values="value",
        aggfunc="first"
    ).sort_index()

    # item_list 순서대로 정렬
    pivot_df = pivot_df.reindex(columns=item_list)

    return pivot_df

rename_fn = {
    '매출액(천원)': '매출액',
    '매출총이익(천원)': '매출총이익',
    '영업이익(천원)': '영업이익',
    '계속사업이익(천원)': '계속사업이익',
    '당기순이익(천원)': '당기순이익',
    '총자산(천원)': '자산총계',
    '유동자산(천원)': '유동자산',
    '재고자산(천원)': '재고자산',
    '총자본(천원)': '자본총계',
    '배당금지급(영업,투자,재무)(천원)': '배당금',
}

rename_fs = {
    '매출액': '매출액',
    '매출총이익': '매출총이익',
    '영업이익': '영업이익',
    '계속사업이익': '계속사업이익',
    '당기순이익': '당기순이익',
    '자산총계': '자산총계',
    '유동자산': '유동자산',
    '재고자산': '재고자산',
    '자본총계': '자본총계',
    '배당금지급': '배당금',    # ← 여기서 배당금지급을 배당금으로 통일
}

In [3]:
def build_merged_df_resize_for_ticker(
    db_info: dict,
    ticker: str,
    item_list: list,
    table_name_dart: str = "korea_fs_data_from_DART",
    table_name_fn: str = "korea_fs_data",
) -> pd.DataFrame:
    """
    1개 ticker에 대해
    - DART 재무제표 + korea_fs_data(천원 단위)를 합쳐서
    - merged_df_resize (TTM, ROE, ROA, payout_ratio 포함) 를 반환.

    [주요 수정 사항]
    - 컬럼명 할당 시 실제 존재하는 컬럼 수에 맞게 안전하게 처리
    - 누락된 컬럼은 NaN으로 채움
    """
    symbol = "A" + ticker  # korea_fs_data 쪽 symbol 형식

    # ---------------- 1) DART 재무제표 ----------------
    df = fetch_fs_data_by_ticker(db_info, ticker, table_name=table_name_dart)
    if df.empty:
        raise ValueError("DART 재무제표가 없습니다.")

    df = df.copy()
    df["report_date"] = pd.to_datetime(df["report_date"])

    # 1-1) 손익 계정 추출
    cont_ops_df = extract_is_group(df, "continuing_operations", account_groups)
    tax_df = extract_is_group(df, "income_tax", account_groups)
    rev_df = df[df["account_id"].isin(account_groups["revenue"])]
    gp_df = df[df["account_id"].isin(account_groups["gross_profit"])]
    op_df = df[df["account_id"].isin(account_groups["operating_income"])]

    # 1-2) pivot + 이름 통일 + Q4 조정
    rev_pivot = make_pivot(rev_df, "매출액")
    gp_pivot = make_pivot(gp_df, "매출총이익")
    op_pivot = make_pivot(op_df, "영업이익")
    cont_ops_pivot = make_pivot(cont_ops_df, "계속사업이익")
    tax_pivot = make_pivot(tax_df, "법인세비용")

    # 각 pivot이 비어있지 않은지 확인
    pivots = {
        "매출액": rev_pivot,
        "매출총이익": gp_pivot,
        "영업이익": op_pivot,
        "계속사업이익": cont_ops_pivot,
        "법인세비용": tax_pivot,
    }

    # 비어있지 않은 pivot만 concat
    valid_pivots = [p for p in pivots.values() if not p.empty]

    if not valid_pivots:
        raise ValueError(f"ticker {ticker}: 모든 손익계정이 비어있습니다.")

    is_table = pd.concat(valid_pivots, axis=1)
    is_table.index.name = "report_date"

    # 비슷한 계정명 병합
    is_table = merge_similar_columns_smart(is_table)

    value_cols = is_table.columns.tolist()
    is_table_adj = adjust_quarterly_q4_only(is_table, value_cols)

    # ===== 핵심 수정: 컬럼명 안전하게 할당 =====
    # 실제 존재하는 컬럼 수에 맞게 이름 부여
    expected_cols = [
        "매출액",
        "매출총이익",
        "영업이익",
        "법인세비용차감전순이익",
        "법인세비용",
    ]

    actual_col_count = len(is_table_adj.columns)

    # 실제 컬럼 수만큼만 이름 할당
    if actual_col_count <= len(expected_cols):
        is_table_adj.columns = expected_cols[:actual_col_count]
    else:
        # 컬럼이 더 많으면 번호를 붙여서 구분
        col_names = expected_cols + [f"기타_{i}" for i in range(actual_col_count - len(expected_cols))]
        is_table_adj.columns = col_names[:actual_col_count]

    # 필수 컬럼이 없으면 생성 (NaN으로 채움)
    required_is_cols = ["매출액", "매출총이익", "영업이익", "법인세비용차감전순이익", "법인세비용"]
    for col in required_is_cols:
        if col not in is_table_adj.columns:
            is_table_adj[col] = np.nan

    # 당기순이익 계산
    is_table_adj["당기순이익"] = (
        is_table_adj["법인세비용차감전순이익"] - is_table_adj["법인세비용"]
    )

    # ===== BS pivot (재무상태표) =====
    assets_df = extract_bs_group(df, "assets_total", account_groups_bs)
    cash_df = extract_bs_group(df, "cash", account_groups_bs)
    current_assets_df = extract_bs_group(df, "current_assets", account_groups_bs)
    inventories_df = extract_bs_group(df, "inventories", account_groups_bs)
    liabilities_df = extract_bs_group(df, "liabilities_total", account_groups_bs)
    current_liab_df = extract_bs_group(df, "current_liabilities", account_groups_bs)
    noncurrent_liab_df = extract_bs_group(df, "noncurrent_liabilities", account_groups_bs)

    assets_pivot = make_pivot(assets_df, "자산총계")
    cash_pivot = make_pivot(cash_df, "현금및현금성자산")
    current_assets_pivot = make_pivot(current_assets_df, "유동자산")
    inventories_pivot = make_pivot(inventories_df, "재고자산")
    liabilities_pivot = make_pivot(liabilities_df, "부채총계")
    current_liab_pivot = make_pivot(current_liab_df, "유동부채")
    noncurrent_liab_pivot = make_pivot(noncurrent_liab_df, "비유동부채")

    bs_pivots = [
        assets_pivot,
        cash_pivot,
        current_assets_pivot,
        inventories_pivot,
        liabilities_pivot,
        current_liab_pivot,
        noncurrent_liab_pivot,
    ]

    valid_bs_pivots = [p for p in bs_pivots if not p.empty]

    if valid_bs_pivots:
        bs_table = pd.concat(valid_bs_pivots, axis=1)

        expected_bs_cols = [
            "자산총계",
            "현금및현금성자산",
            "유동자산",
            "재고자산",
            "부채총계",
            "유동부채",
            "비유동부채",
        ]

        actual_bs_count = len(bs_table.columns)
        bs_table.columns = expected_bs_cols[:actual_bs_count]

        # 필수 BS 컬럼 생성
        required_bs_cols = ["자산총계", "부채총계"]
        for col in required_bs_cols:
            if col not in bs_table.columns:
                bs_table[col] = np.nan

        # 자본총계 = 자산총계 - 부채총계
        bs_table["자본총계"] = bs_table["자산총계"] - bs_table["부채총계"]
    else:
        # BS 데이터가 없으면 빈 DataFrame
        bs_table = pd.DataFrame(index=is_table_adj.index)
        bs_table["자산총계"] = np.nan
        bs_table["부채총계"] = np.nan
        bs_table["자본총계"] = np.nan

    # ===== CF pivot (현금흐름표) =====
    cf_op_df = extract_cf_group(df, "cf_operating", account_groups_cf)
    cf_inv_df = extract_cf_group(df, "cf_investing", account_groups_cf)
    cf_fin_df = extract_cf_group(df, "cf_financing", account_groups_cf)
    cf_tax_df = extract_cf_group(df, "cf_tax_operating", account_groups_cf)
    cf_div_df = extract_cf_group(df, "cf_dividends_paid", account_groups_cf)

    cf_op_pivot = make_pivot(cf_op_df, "영업활동으로인한현금흐름")
    cf_inv_pivot = make_pivot(cf_inv_df, "투자활동으로인한현금흐름")
    cf_fin_pivot = make_pivot(cf_fin_df, "재무활동으로인한현금흐름")
    cf_tax_pivot = make_pivot(cf_tax_df, "법인세")
    cf_div_pivot = make_pivot(cf_div_df, "배당금")

    cf_pivots = [cf_op_pivot, cf_inv_pivot, cf_fin_pivot, cf_tax_pivot, cf_div_pivot]
    valid_cf_pivots = [p for p in cf_pivots if not p.empty]

    if valid_cf_pivots:
        cf_table = pd.concat(valid_cf_pivots, axis=1)
        value_cols_cf = cf_table.columns.tolist()
        cf_table = cumulative_to_quarterly(cf_table, value_cols_cf)
    else:
        cf_table = pd.DataFrame(index=is_table_adj.index)
        cf_table["배당금"] = np.nan

    # DART 기준 전체 fs_df
    fs_df = pd.concat([is_table_adj, bs_table, cf_table], axis=1)
    fs_df.index.name = "date"
    fs_df.index = pd.to_datetime(fs_df.index)
    fs_df = fs_df.sort_index()

    # 천원단위 맞추기
    fs_df_scaled = fs_df / 1000.0

    # ---------------- 2) korea_fs_data (fn_df) ----------------
    try:
        fn_raw = fetch_fs_from_db(db_info, symbol, table_name=table_name_fn)
        fn_df = make_fs_pivot(fn_raw, item_list)
        fn_df_u = fn_df.rename(columns=rename_fn)
    except Exception as e:
        print(f"[WARN] korea_fs_data 로드 실패 (ticker={ticker}): {e}")
        fn_df_u = pd.DataFrame()

    fs_df_u = fs_df_scaled.rename(columns=rename_fs)

    # 공통 컬럼
    common_cols = [
        "매출액",
        "매출총이익",
        "영업이익",
        "계속사업이익",
        "당기순이익",
        "자산총계",
        "유동자산",
        "재고자산",
        "자본총계",
        "배당금",
    ]

    if not fn_df_u.empty:
        fn_df_u = fn_df_u.sort_index()
        fs_df_u = fs_df_u.sort_index()
        fn_df_u = fn_df_u[[c for c in common_cols if c in fn_df_u.columns]]
        fs_df_u = fs_df_u[[c for c in common_cols if c in fs_df_u.columns]]

        all_dates = fn_df_u.index.union(fs_df_u.index)
        fn_all = fn_df_u.reindex(all_dates)
        fs_all = fs_df_u.reindex(all_dates)

        merged_df = fn_all.combine_first(fs_all)
    else:
        merged_df = fs_df_u

    merged_df = merged_df[[c for c in common_cols if c in merged_df.columns]]

    # 필수 컬럼 확인 및 생성
    required_cols = ["매출액", "매출총이익", "영업이익", "당기순이익", "자산총계", "자본총계", "배당금"]
    for col in required_cols:
        if col not in merged_df.columns:
            merged_df[col] = np.nan

    # 필요한 컬럼만 축소
    merged_df_resize = merged_df[
        ["매출액", "매출총이익", "영업이익", "당기순이익", "자산총계", "자본총계", "배당금"]
    ].copy()

    # TTM 계산
    merged_df_resize["매출총이익_ttm"] = merged_df_resize["매출총이익"].rolling(4).sum()
    merged_df_resize["당기순이익_ttm"] = merged_df_resize["당기순이익"].rolling(4).sum()
    merged_df_resize["영업이익_ttm"] = merged_df_resize["영업이익"].rolling(4).sum()
    merged_df_resize["매출액_ttm"] = merged_df_resize["매출액"].rolling(4).sum()

    # lag & 평균 자본/자산
    merged_df_resize["자본총계_lag4"] = merged_df_resize["자본총계"].shift(4)
    merged_df_resize["자산총계_lag4"] = merged_df_resize["자산총계"].shift(4)
    merged_df_resize["자본총계_평균"] = (
        merged_df_resize["자본총계"] + merged_df_resize["자본총계_lag4"]
    ) / 2
    merged_df_resize["자산총계_평균"] = (
        merged_df_resize["자산총계"] + merged_df_resize["자산총계_lag4"]
    ) / 2

    # 마진, ROA, ROE, payout
    merged_df_resize["GPM_ttm"] = safe_divide(
        merged_df_resize["매출총이익_ttm"], merged_df_resize["매출액_ttm"]
    )
    merged_df_resize["OPM_ttm"] = safe_divide(
        merged_df_resize["영업이익_ttm"], merged_df_resize["매출액_ttm"]
    )
    merged_df_resize["NIM_ttm"] = safe_divide(
        merged_df_resize["당기순이익_ttm"], merged_df_resize["매출액_ttm"]
    )
    merged_df_resize["ROA"] = safe_divide(
        merged_df_resize["당기순이익_ttm"], merged_df_resize["자산총계_평균"]
    )
    merged_df_resize["ROE"] = safe_divide(
        merged_df_resize["당기순이익_ttm"], merged_df_resize["자본총계_평균"]
    )
    merged_df_resize["payout_ratio"] = safe_divide(
        merged_df_resize["배당금"], merged_df_resize["당기순이익_ttm"]
    )

    return merged_df_resize

In [6]:
def build_long_for_all_tickers(
    db_info: dict,
    item_list: list,
    table_name_dart: str = "korea_fs_data_from_DART",
    table_name_fn: str = "korea_fs_data",
):
    """
    1) DART 테이블에서 unique ticker 리스트를 가져오고
    2) 각 ticker마다 merged_df_resize를 계산한 뒤
    3) date, ticker, indicator, value long-format으로 변환하여
       하나의 DataFrame으로 concat.
    4) 중간 에러는 error_list에 (ticker, msg) 형태로 기록.
    """
    # ticker_list = get_unique_tickers(db_info, table_name=table_name_dart)
    ticker_list = ['000660', '005930', '005380']
    print(f"[INFO] 총 {len(ticker_list)}개 ticker 처리 예정")

    all_long = []
    error_list = []

    for tkr in ticker_list:
        try:
            print(f"[RUN] ticker = {tkr}")

            merged_df_resize = build_merged_df_resize_for_ticker(
                db_info=db_info,
                ticker=tkr,
                item_list=item_list,
                table_name_dart=table_name_dart,
                table_name_fn=table_name_fn,
            )

            long_df = df_to_long_format(merged_df_resize, ticker=tkr)
            all_long.append(long_df)

        except Exception as e:
            msg = str(e)
            print(f"[ERROR] ticker={tkr} : {msg}")
            error_list.append((tkr, msg))
            # 에러가 나도 다음 ticker 계속 진행
            continue

    if all_long:
        result_long = pd.concat(all_long, ignore_index=True)
    else:
        # 아무것도 성공 못했을 때 빈 df
        result_long = pd.DataFrame(columns=["date", "ticker", "indicator", "value"])

    return result_long, error_list


In [7]:
db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user' : 'stox7412',
    'password' : 'Apt106503!~',
    'database': 'investar'
}

item_list = [
    '매출액(천원)', '매출총이익(천원)', '영업이익(천원)',
    '계속사업이익(천원)', '당기순이익(천원)',
    '총자산(천원)', '유동자산(천원)', '재고자산(천원)',
    '총자본(천원)', '배당금지급(영업,투자,재무)(천원)'
]

# 위에서 정의한 db_info, item_list 를 그대로 사용한다고 가정
all_long_df, error_list = build_long_for_all_tickers(
    db_info=db_info,
    item_list=item_list,
    table_name_dart="korea_fs_data_from_DART",
    table_name_fn="korea_fs_data",
)

print(all_long_df.head())
print("에러 ticker 리스트:", error_list)

[INFO] 총 3개 ticker 처리 예정
[RUN] ticker = 000660
[RUN] ticker = 005930
[RUN] ticker = 005380
        date  ticker indicator         value
0 2004-03-31  000660       매출액  1.296647e+09
1 2004-06-30  000660       매출액  1.683512e+09
2 2004-09-30  000660       매출액  1.542350e+09
3 2004-12-31  000660       매출액  1.341844e+09
4 2005-03-31  000660       매출액  1.284277e+09
에러 ticker 리스트: []


In [20]:
def create_roe_roa_table_if_not_exists(db_info: dict,
                                       table_name: str = "korea_fs_data_roe_roa"):
    """
    date, ticker, indicator, value 구조의 테이블을 생성.
    PK(date, ticker, indicator) 로 중복 시 value를 UPDATE 가능.
    """
    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4"
    )

    try:
        with conn.cursor() as cur:
            create_sql = f"""
            CREATE TABLE IF NOT EXISTS {table_name} (
                date DATE NOT NULL,
                ticker VARCHAR(20) NOT NULL,
                indicator VARCHAR(100) NOT NULL,
                value DOUBLE NULL,
                PRIMARY KEY (date, ticker, indicator)
            ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
            """
            cur.execute(create_sql)
            conn.commit()
    finally:
        conn.close()


def upsert_roe_roa_long_df(long_df: pd.DataFrame,
                           db_info: dict,
                           table_name: str = "korea_fs_data_roe_roa",
                           batch_size: int = 1000) -> None:
    """
    long_format(df: [date, ticker, indicator, value])를
    korea_fs_data_roe_roa 테이블에 INSERT ... ON DUPLICATE KEY UPDATE 로 저장.

    - (date, ticker, indicator)가 같으면 value를 덮어씀.
    - batch_size 단위로 나누어 executemany 수행.
    """

    # 0) 컬럼 체크
    required_cols = {"date", "ticker", "indicator", "value"}
    if not required_cols.issubset(long_df.columns):
        missing = required_cols - set(long_df.columns)
        raise ValueError(f"long_df에 필요한 컬럼이 없습니다: {missing}")

    # 1) date를 datetime → date 로 정리
    df = long_df.copy()

    # date가 index 인 경우도 방지
    if "date" not in df.columns and df.index.name == "date":
        df = df.reset_index()

    # 문자열이면 datetime 으로
    if not np.issubdtype(df["date"].dtype, np.datetime64):
        df["date"] = pd.to_datetime(df["date"])

    # pymysql 에는 python date 객체로 넣는 것이 안전
    df["date"] = df["date"].dt.date

    # ticker, indicator 문자열로 정리
    df["ticker"] = df["ticker"].astype(str)
    df["indicator"] = df["indicator"].astype(str)

    # value는 float (NaN 은 None 으로)
    df["value"] = df["value"].astype(float)

    # 2) records 리스트 생성
    records = []
    for _, row in df.iterrows():
        records.append((
            row["date"],
            row["ticker"],
            row["indicator"],
            None if pd.isna(row["value"]) else float(row["value"])
        ))

    if not records:
        print("[INFO] 저장할 레코드가 없습니다.")
        return

    # 3) DB 연결 및 upsert
    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4",
        autocommit=False
    )

    try:
        with conn.cursor() as cur:
            # 테이블 없으면 생성
            create_roe_roa_table_if_not_exists(db_info, table_name)

            insert_sql = f"""
            INSERT INTO {table_name} (date, ticker, indicator, value)
            VALUES (%s, %s, %s, %s)
            ON DUPLICATE KEY UPDATE
                value = VALUES(value)
            """

            # 배치로 insert
            total = len(records)
            for start in range(0, total, batch_size):
                end = start + batch_size
                batch = records[start:end]
                cur.executemany(insert_sql, batch)
                conn.commit()
                print(f"[INFO] {start} ~ {end-1} 행까지 커밋 완료")

        print(f"[INFO] 총 {len(records)} 개 row upsert 완료")

    finally:
        conn.close()


In [21]:
# 예: all_long_list 에 ticker별 long_df 를 append 해 두었다고 가정
# all_long_df = pd.concat(all_long_list, ignore_index=True)

# DB에 upsert 저장
upsert_roe_roa_long_df(
    long_df=all_long_df,
    db_info=db_info,
    table_name="korea_fs_data_roe_roa",
    batch_size=1000
)

OperationalError: (2003, "Can't connect to MySQL server on '192.168.0.230' ([WinError 10061] 대상 컴퓨터에서 연결을 거부했으므로 연결하지 못했습니다)")

In [5]:
import pymysql
import pandas as pd

def count_unique_tickers(db_info: dict,
                         table_name: str = "korea_fs_data_roe_roa") -> int:
    """
    해당 테이블에서 unique한 ticker 개수를 반환하는 함수.
    """
    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4"
    )

    try:
        query = f"SELECT DISTINCT ticker FROM {table_name};"
        df = pd.read_sql(query, conn)

        unique_count = df['ticker'].nunique()

        print(f"[INFO] Unique ticker count in {table_name}: {unique_count}")
        return unique_count

    finally:
        conn.close()

In [6]:
db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user' : 'stox7412',
    'password' : 'Apt106503!~',
    'database': 'investar'
}



count_unique_tickers(db_info)

[INFO] Unique ticker count in korea_fs_data_roe_roa: 699


699

In [ ]:
# 예: 새로 계산한 single_long_df (한 종목 또는 몇 종목 데이터)
# upsert_roe_roa_long_df(single_long_df, db_info, "korea_fs_data_roe_roa")
# → 같은 (date, ticker, indicator)가 있으면 value만 업데이트됨